# phenoforge evaluation

Scores each retrieval method — BM25, dense (BioLORD-2023), hybrid, and hierarchy
expansion — against the curated OHDSI Phenotype Library demo cohorts as ground truth.
Exploration only, never pushed to production; the metrics and orchestration this
notebook calls live in `src/phenoforge/eval/`.

## Why not exact-match recall

Standard recall@k is the wrong metric here. Retrieving `E11.21` when the target is
`E11.9` is a near-miss under the same parent (`E11`, Type 2 diabetes mellitus); retrieving
a circulatory-chapter code for a diabetes query is a total miss. Exact-match recall scores
both as zero. Every metric below is instead derived from **tree distance** — how many
hierarchy edges separate a retrieved code from its nearest ground-truth code — so
near-misses earn partial credit and total misses don't.

## The metrics

- **Tree distance & distance-weighted score.** Distance between two ICD-10-CM codes,
  found via their lowest common ancestor in the `Is a` hierarchy. `distance=0` (exact
  match) scores `1.0`; score decays linearly to `0.0` at `distance >= 6`; no common
  ancestor (different chapters) scores `0.0`.
- **Coverage** (recall-like). For each ground-truth code, its best distance-weighted
  match among predicted codes, averaged — did the whole curated set get assembled?
- **Over-inclusion penalty** (1 − precision-like). For each predicted code, `1 -` its
  best match among ground-truth codes, averaged — how much of what was retrieved has
  no good match anywhere in the curated set. A predicted code close to *something* in
  ground truth still earns partial credit rather than a flat penalty, because the
  curated set is a **lower bound**, not an exact set (see below).
- **Hierarchical score.** Harmonic mean of coverage and `1 - over_inclusion_penalty` —
  a single combined quality number, F1-style. Harmonic rather than arithmetic so a
  method can't hide bad precision behind good coverage or vice versa.

## Ground truth is a lower bound, not an exact set

Curated cohort definitions get resolved from SNOMED to ICD-10-CM (`engine/curated.py`),
and three things are conservatively dropped along the way: excluded items, unexpanded
`includeDescendants` items, and SNOMED concepts whose ICD-10-CM fan-out is too broad to
resolve safely. So a method retrieving more than the curated set isn't automatically
wrong — this is exactly why over-inclusion uses partial credit instead of set membership.

**Not evaluated here:** decomposition accuracy (breaking a population description into
seed concepts) — nothing in the codebase does that yet; it's the not-yet-built LangGraph
agent's job.

In [ ]:
import os
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's own directory, not the repo
# root, so relative data/ paths below would silently look in notebooks/data/.
# Walk up to the directory containing pyproject.toml and chdir there once.
_dir = Path.cwd()
while not (_dir / "pyproject.toml").exists():
    if _dir.parent == _dir:
        raise FileNotFoundError(f"Could not find repo root (pyproject.toml) above {Path.cwd()}")
    _dir = _dir.parent
os.chdir(_dir)
print(f"cwd: {Path.cwd()}")

import json

import matplotlib.pyplot as plt
import pandas as pd

from phenoforge.engine.db import DEFAULT_VOCAB_DB_PATH, connect
from phenoforge.engine.dense import DenseRetriever
from phenoforge.eval.harness import run_benchmark

LIBRARY_DIR = Path("data/phenotype_library")
INDEX_PATH = Path("data/concept_index.lance")

if not DEFAULT_VOCAB_DB_PATH.exists():
    raise FileNotFoundError(
        f"{DEFAULT_VOCAB_DB_PATH} not found. Run: python scripts/load_vocab.py data/athena"
    )
if not (LIBRARY_DIR / "manifest.json").exists():
    raise FileNotFoundError(
        f"{LIBRARY_DIR} not found. Run: python scripts/fetch_phenotype_library.py"
    )

con = connect()
manifest = json.loads((LIBRARY_DIR / "manifest.json").read_text())
print(f"Connected to {DEFAULT_VOCAB_DB_PATH}, {len(manifest)} bundled cohorts")

In [ ]:
# Dense/hybrid only run for real if the index has been built
# (python scripts/build_index.py); otherwise those methods are skipped
# rather than erroring, same fallback search_concepts uses.
dense = DenseRetriever(con, index_path=INDEX_PATH) if INDEX_PATH.exists() else None
methods = ["bm25", "hybrid", "expand_descendants"] + (["dense"] if dense else [])
print(f"Dense index: {'loaded' if dense else 'not built — dense/hybrid-with-dense skipped'}")
print(f"Methods: {methods}")

In [ ]:
report = run_benchmark(con, LIBRARY_DIR, cohort_ids=list(manifest), methods=methods, dense=dense)

summary = pd.DataFrame(
    {
        "coverage": report.mean_coverage_by_method,
        "over_inclusion_penalty": report.mean_over_inclusion_by_method,
        "hierarchical_score": report.mean_hierarchical_score_by_method,
    }
).round(3)
summary

## Method comparison

Grouped bars, one group per metric, one color per method — all three metrics share the
same `[0, 1]` scale, so a single axis is correct here (no dual-axis).

In [ ]:
# Fixed categorical color order (validated colorblind-safe palette,
# adjacent-pairlist gate for bar charts) — never reassigned per filter.
METHOD_COLORS = {
    "bm25": "#2a78d6",
    "dense": "#eb6834",
    "hybrid": "#1baf7a",
    "expand_descendants": "#eda100",
}

metrics = ["coverage", "over_inclusion_penalty", "hierarchical_score"]
methods_present = [m for m in METHOD_COLORS if m in summary.index]

x = range(len(metrics))
width = 0.8 / len(methods_present)

fig, ax = plt.subplots(figsize=(8, 5))
for i, method in enumerate(methods_present):
    offsets = [xi + i * width for xi in x]
    values = summary.loc[method, metrics]
    bars = ax.bar(offsets, values, width=width, label=method, color=METHOD_COLORS[method])
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)

ax.set_xticks([xi + width * (len(methods_present) - 1) / 2 for xi in x])
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel("score")
ax.set_title("Retrieval method comparison against curated ground truth")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Per-cohort detail

Every cohort/method result, sorted worst-to-best on `hierarchical_score` — useful for
spotting which specific cohorts a method struggles with, not just the aggregate mean.

In [ ]:
detail = pd.DataFrame(
    [
        {
            "cohort": r.cohort_name,
            "method": r.method,
            "n_ground_truth": len(r.ground_truth_codes),
            "n_predicted": len(r.predicted_codes),
            "coverage": round(r.coverage, 3),
            "over_inclusion_penalty": round(r.over_inclusion_penalty, 3),
            "hierarchical_score": round(r.hierarchical_score, 3),
        }
        for r in report.results
    ]
).sort_values("hierarchical_score")
detail